# CRISP-DM Fase 1: Business Understanding (Comprensión del Negocio)
## Dataset: SIPSA Precios Mayoristas (DANE)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 1: Business Understanding) | **Fase PDCO**: PLAN  
**Estándares Aplicados**: SWEBOK Cap. 1 (Requerimientos de Software), DAMA-DMBOK 2 (Metadata & Governance), ISO/IEC 25010.

---

### 1. Contexto Estratégico y Problema de Negocio
La formación de precios en mercados mayoristas sigue procesos estocásticos con alta sensibilidad a shocks climáticos y de oferta. Se contrastan modelos gaussianos estándar con estimadores robustos inmunes a valores atípicos.

**Objetivo de Negocio**:
> Identificar patrones de volatilidad, shocks de precios y tendencias estacionales en plazas de mercado mayoristas para la toma de decisiones comerciales y mitigación de riesgo.

### 2. Entidad del Dominio y Alcance
* **Entidad Analítica Principal**: `Cotizaciones Mayoristas Diarias de Productos Agrícolas (COP/Kg)`
* **Custodio Oficial de los Datos**: DANE - Sistema de Información de Precios y Abastecimiento del Sector Agropecuario (SIPSA)
* **Granularidad Espacial**: `Nodo Urbano: Central Mayorista / Terminal de Abastos`
* **Granularidad Temporal**: `Diaria (Lunes a Viernes)`

### 3. Preguntas de Negocio Mapeadas (`docs/1-Bateria_preguntas.md`):
* **E1: Volatilidad de Precios**
* **E2: Tendencia de Precio**
* **F2: Concentración de Precios Altos**

### 4. Indicadores Clave de Desempeño (KPIs):
* `Precio medio y mediana robusta`
* `Coeficiente de Variación (CV)`
* `Pendiente de tendencia Theil-Sen`

---

### 📋 FICHA TÉCNICA DEL DATASET (DAMA-DMBOK 2)

#### 📦 A. QUÉ HABÍA EN EL DATASET ORIGINAL (Diagnóstico Raw)
> **Estado de Origen**: Cotizaciones diarias por plaza mayorista y producto. Tablas desestructuradas con vacíos por días festivos, empaques variados (canastilla, bulto 50kg, atado), y shocks de precios atípicos por paros o heladas.

#### 🔧 B. QUÉ SE NECESITA PARA CONSTRUIR LOS INDICADORES (Transformaciones y Fórmulas)
> **Requisitos y Formulación**: Normalización de precios a COP/kg, imputación de fines de semana por spline/Theil-Sen, depuración de atípicos (1.5xIQR), Coeficiente de Variación (CV = sigma/mu) y descomposición estacional STL.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN ROBUSTA DE RUTAS DEL PROYECTO (PEP 8)]
# Compatible con Local Jupyter, VSCode, Colab, Databricks Git Folders & Repos
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Búsqueda dinámica ascendente de la raíz del proyecto (donde reside src/ y metadata.json)
CURRENT_DIR = Path(".").resolve()
APP_ROOT = None

for candidate in [CURRENT_DIR, CURRENT_DIR.parent, CURRENT_DIR.parent.parent, CURRENT_DIR.parent.parent.parent]:
    if (candidate / "src" / "database").exists() or (candidate / "metadata.json").exists():
        APP_ROOT = candidate
        break

if APP_ROOT is None:
    candidate = CURRENT_DIR
    while candidate.parent != candidate:
        if (candidate / "src" / "database").exists():
            APP_ROOT = candidate
            break
        candidate = candidate.parent

if APP_ROOT is None:
    APP_ROOT = CURRENT_DIR

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno e importaciones inicializadas exitosamente.")
print(f"📍 Raíz del proyecto detectada: {APP_ROOT}")


### 5. Fundamentación del Contrato de Datos (Data Contract)
Bajo los lineamientos de **DAMA-DMBOK 2**, todo pipeline de datos analíticos debe iniciar con la verificación de un **Contrato de Datos**. 
El contrato define formalmente:
1. El custodio del dato y la periodicidad de actualización.
2. La definición operativa de las variables críticas y sus tipos de datos esperados.
3. Las tolerancias de calidad (porcentaje máximo de nulos y umbrales de detección de valores anómalos).


In [ ]:
# ==============================================================================
# [FASE 1: VERIFICACIÓN DEL CONTRATO DE NEGOCIO Y ESPECIFICACIÓN TÉCNICA]
# ==============================================================================
import yaml

config_path = APP_ROOT / "config" / "datasets_config.yaml"
if config_path.exists():
    with open(config_path, "r", encoding="utf-8") as f:
        config_data = yaml.safe_load(f)
    print(f"📄 Archivo de configuración cargado: {config_path.name}")
    dataset_cfg = config_data.get("datasets", {}).get("02_sipsa_precios", {})
    print(f"📋 Especificación del Dataset en YAML: {dataset_cfg}")
else:
    print("ℹ️ Configuración YAML por defecto activa.")

print("\n--- DEFINICIÓN FORMAL DEL CONTRATO ---")
print("• Entidad:", "Cotizaciones Mayoristas Diarias de Productos Agrícolas (COP/Kg)")
print("• Custodio:", "DANE - Sistema de Información de Precios y Abastecimiento del Sector Agropecuario (SIPSA)")
print("• Granularidad Espacial:", "Nodo Urbano: Central Mayorista / Terminal de Abastos")
print("• Granularidad Temporal:", "Diaria (Lunes a Viernes)")
print("• KPIs de Negocio:", ['Precio medio y mediana robusta', 'Coeficiente de Variación (CV)', 'Pendiente de tendencia Theil-Sen'])


### 6. Verificación de Disponibilidad en el Data Lakehouse
El Lakehouse implementa la arquitectura Medallion:
* **Bronze**: Datos crudos inmutables preservando el formato original del custodio.
* **Silver**: Datos limpios, sanitizados y estructurados en formato columnar Parquet comprimido con Snappy.
* **Gold**: Base de datos relacional SQLite (`agrostats_lakehouse.db`) con esquemas conformados e índices para consultas analíticas de alta velocidad.


In [ ]:
# ==============================================================================
# [FASE 1: INSPECCIÓN DE DISPONIBILIDAD EN LAS CAPAS DEL LAKEHOUSE]
# ==============================================================================
from src.database.db_manager import DatabaseManager

db_path = APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"
parquet_path = APP_ROOT / "data/processed/sipsa_precios.parquet"

print("🔍 Verificación de Almacenamiento en Data Lakehouse:")
print(f"• Capa Silver (Parquet): {parquet_path.name} -> Existe: {parquet_path.exists()}")
print(f"• Capa Gold (SQLite DB): {db_path.name} -> Existe: {db_path.exists()}")

if parquet_path.exists():
    df_sample = pd.read_parquet(parquet_path)
    print(f"\n✅ Contrato de datos preliminar verificado exitosamente:")
    print(f"• Total de registros curados: {len(df_sample):,} filas")
    print(f"• Columnas disponibles: {list(df_sample.columns)}")


### 7. Síntesis y Transición a la Fase 2 (Data Understanding)
* **Entregable de la Fase 1**: Especificación formal del problema de negocio, KPIs concertados y contrato de datos validado.
* **Siguiente Paso**: Proceder al notebook `02_ingestion.ipynb` para ejecutar la extracción inmutable y auditoría criptográfica SHA-256.
